In [2]:
import os
import requests
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # finds GenAI/LLM/.env by searching parent directories
OPEN_ROUTER_API_KEY = os.environ["OPEN_ROUTER_API_KEY"]

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {OPEN_ROUTER_API_KEY}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "moonshotai/kimi-k2.5",
    "messages": [
        {
          "role": "user",
          "content": "How many r's are in the word 'strawberry'?"
        }
      ],
    "reasoning": {"enabled": True},
    "max_tokens": 2048  # kept low to fit within available OpenRouter credits
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
if 'choices' not in response:
    raise RuntimeError(f"OpenRouter request failed: {response}")
response = response['choices'][0]['message']
print("First response:", response.get('content'))

# Preserve the assistant message with reasoning_details
messages = [
  {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
  {
    "role": "assistant",
    "content": response.get('content'),
    "reasoning_details": response.get('reasoning_details')  # Pass back unmodified
  },
  {"role": "user", "content": "Are you sure? Think carefully."}
]

# Second API call - model continues reasoning from where it left off
response2 = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {OPEN_ROUTER_API_KEY}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "moonshotai/kimi-k2.5",
    "messages": messages,  # Includes preserved reasoning_details
    "reasoning": {"enabled": True},
    "max_tokens": 2048
  })
)
response2 = response2.json()
if 'choices' not in response2:
    raise RuntimeError(f"OpenRouter request failed: {response2}")
print("Second response:", response2['choices'][0]['message']['content'])

First response: There are **3** r's in the word "strawberry."

Here is the breakdown: s-t-**r**-a-w-b-e-**r**-**r**-y.
Second response: Yes, I am sure. Let me verify once more by going through each letter:

**S** - T - **R** - A - W - B - E - **R** - **R** - Y

1. The 3rd letter: **r**
2. The 8th letter: **r**
3. The 9th letter: **r**

There are indeed **3 r's** in "strawberry."
